In [ ]:
# %% Deep learning - Section 21.199
#    Code challenge 34: VGG-16
#
#    1) Start from code from video 21.198 (ResNet-18 on STL10)
#    2) Fine-tune using VGG-16 instead of ResNet-18
#    3) Keep the code as similar as possible but change what you need to change
#    4) Compare to ResNet-18

# This code pertains a deep learning course provided by Mike X. Cohen on Udemy:
#   > https://www.udemy.com/course/deeplearning_x
# The "base" code in this repository is adapted (with very minor modifications)
# from code developed by the course instructor (Mike X. Cohen), while the
# "exercises" and the "code challenges" contain more original solutions and
# creative input from my side. If you are interested in DL (and if you are
# reading this statement, chances are that you are), go check out the course, it
# is singularly good.

In [1]:
# %% Libraries and modules
import numpy                  as np
import matplotlib.pyplot      as plt
import torch
import torch.nn               as nn
import seaborn                as sns
import copy
import torch.nn.functional    as F
import pandas                 as pd
import scipy.stats            as stats
import sklearn.metrics        as skm
import time
import sys
import imageio.v2             as imageio
import torchvision
import torchvision.transforms as T

from torch.utils.data                 import DataLoader,TensorDataset,Dataset,Subset
from sklearn.model_selection          import train_test_split
from google.colab                     import files
from torchsummary                     import summary
from scipy.stats                      import zscore
from sklearn.decomposition            import PCA
from scipy.signal                     import convolve2d
from torchsummary                     import summary
from IPython                          import display
from matplotlib_inline.backend_inline import set_matplotlib_formats
set_matplotlib_formats('svg')
plt.style.use('default')


In [ ]:
# %% Use GPU

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print(device)


In [ ]:
# %% Data

# We'll try the ResNet on a dataset on which it was not trained (STL10). Since
# ResNet is trained for images in a specific range (not [-1,1]), the mean and
# std normalisation values in the transform need to change

# Transformations (vals can be found online)
mean = [0.485, 0.456, 0.406]
std  = [0.229, 0.224, 0.225]

# Reacll that .ToTensor() normalises to range [0,1]
transform = T.Compose([ T.ToTensor(),
                        T.Normalize(mean=mean,std=std) ])

# Import data and apply the transform
trainset = torchvision.datasets.STL10(root='./data', download=True, split='train', transform=transform)
testset  = torchvision.datasets.STL10(root='./data', download=True, split='test',  transform=transform)

# Convert into DataLoader objects
batchsize    = 32
train_loader = DataLoader(trainset,batch_size=batchsize,shuffle=True,drop_last=True)
test_loader  = DataLoader(testset,batch_size=256)


In [ ]:
# %% Inspect dataset

# Check out the shape of the datasets
print('Data shapes (train/test):')
print( trainset.data.shape )
print( testset.data.shape )

# Check out the range of pixel intensity values
print('\nData value range:')
print( (np.min(trainset.data),np.max(trainset.data)) )

# Check out the unique categories
print('\nData categories:')
print( trainset.classes )


In [ ]:
# %% Some apparent issues

# It looks like the images are not normalised !

# But by now you know the transform is applied when calling the DataLoaders ...
X,y = next(iter(train_loader))

# Try again
print('Data shapes (train/test):')
print( X.data.shape )

# See range of pixel intensity values (now normalised)
print('\nData value range:')
print( (torch.min(X.data),torch.max(X.data)) )


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig,axs = plt.subplots(4,4,figsize=(phi*6,6))

for (i,ax) in enumerate(axs.flatten()):

    # Extract an image and transpose back to [96x96x3], for visualisation also
    # undo the normalisation
    pic = X.data[i].numpy().transpose((1,2,0))
    pic = pic-np.min(pic)
    pic = pic/np.max(pic)

    label = trainset.classes[y[i]]

    ax.imshow(pic)
    ax.text(0,0,label,ha='left',va='top',fontweight='bold',color='k',backgroundcolor='y')
    ax.axis('off')

plt.tight_layout()

plt.savefig('figure18_code_challenge_34.png')
plt.show()
files.download('figure18_code_challenge_34.png')


In [ ]:
# %% Import VGG-16

vgg = torchvision.models.vgg16(pretrained=True)
print(vgg)


In [ ]:
# %% Model's summary

vgg.to(device);
summary(vgg,(3,96,96))


In [ ]:
# %% Freeze layers and modify the model

# Freeze all layers (feature block and first 2 classifier layers)
for p in vgg.parameters():
    p.requires_grad = False

# Print classifier layers (check n_layer and n_nodes)
print(vgg.classifier)

# Change and unfreeze final layer
vgg.classifier[6] = nn.Linear(4096,10)

for p in vgg.classifier[6].parameters():
    p.requires_grad = True

# Ship to GPU
vgg.to(device);


In [9]:
# %% Define loss function and optimizer

loss_fun  = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(vgg.classifier[6].parameters(),lr=0.001,momentum=.9)


In [ ]:
# %% Train the model

# Takes ~4 mins overall with GPU
numepochs = 10

# Preallocate variables
train_loss = []
test_loss  = []
train_acc  = []
test_acc   = []

# loop over epochs
for epochi in range(numepochs):

    # Compute time
    start = time.time()

    # Loop over training data batches
    batch_loss = []
    batch_acc  = []

    for X,y in train_loader:

        # Ship data to GPU
        X = X.to(device)
        y = y.to(device)

        # Forward propagation and loss
        yHat = vgg(X)
        loss = loss_fun(yHat,y)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # Loss and accuracy from current batch
        batch_loss.append( loss.item() )
        batch_acc.append( torch.mean((torch.argmax(yHat,axis=1) == y).float()).item() )

    # Average losses and accuracies across the batches
    train_loss.append( np.mean(batch_loss) )
    train_acc.append( 100*np.mean(batch_acc) )

    # Test performance (here done in batches)
    vgg.eval()

    batch_acc  = []
    batch_loss = []

    for X,y in test_loader:

        # Ship data to GPU
        X = X.to(device)
        y = y.to(device)

        # Forward propagation and loss
        with torch.no_grad():
            yHat = vgg(X)
            loss = loss_fun(yHat,y)

        # Loss and accuracy from current batch
        batch_loss.append(loss.item())
        batch_acc.append( torch.mean((torch.argmax(yHat,axis=1) == y).float()).item() )

    vgg.train()

    # Average losses and accuracies across the batches
    test_loss.append( np.mean(batch_loss) )
    test_acc.append( 100*np.mean(batch_acc) )

    # Print status update
    print(f'Finished epoch {epochi+1}/{numepochs}. Test accuracy = {test_acc[epochi]:.2f}%')
    print(f'Time elapsed: {(time.time()-start)/60:.2f} min\n')


In [ ]:
# %% Plotting

phi = (1 + np.sqrt(5)) / 2
fig,ax = plt.subplots(1,2,figsize=(1.5*phi*5,5))

ax[0].plot(train_loss,'o-',label='Train')
ax[0].plot(test_loss,'o-',label='Test')
ax[0].set_xlabel('Epochs')
ax[0].set_ylabel('Loss (MSE)')
ax[0].set_title('Model loss')
ax[0].legend()

ax[1].plot(train_acc,'o-',label='Train')
ax[1].plot(test_acc,'o-',label='Test')
ax[1].set_xlabel('Epochs')
ax[1].set_ylabel('Accuracy (%)')
ax[1].set_title(f'Final model train/test accuracy: {train_acc[-1]:.2f}/{test_acc[-1]:.2f}%')
ax[1].legend()

plt.suptitle('Pretrained VGG-16 on STL10 data',fontweight='bold',fontsize=14)

plt.savefig('figure19_code_challenge_34.png')
plt.show()
files.download('figure19_code_challenge_34.png')


In [ ]:
# %% Plotting

# Inspect a few random images
X,y = next(iter(test_loader))
X   = X.to(device)
y   = y.to(device)

vgg.eval()
predictions = torch.argmax(vgg(X),axis=1)

phi = (1 + np.sqrt(5)) / 2
fig,axs = plt.subplots(4,4,figsize=(phi*6,6))

for (i,ax) in enumerate(axs.flatten()):

    # Extract an image and transpose back to [32x32x3], for visualisation also
    # undo the normalisation
    pic = X.data[i].cpu().numpy().transpose((1,2,0))
    pic = pic-np.min(pic)
    pic = pic/np.max(pic)

    ax.imshow(pic)

    label = trainset.classes[predictions[i]]
    truec = trainset.classes[y[i]]
    title = f'True: {truec} - Pred: {label}'

    # Title with color-coded accuracy
    titlecolor = 'g' if truec==label else 'r'
    ax.text(48,90,title,ha='center',va='top',fontweight='bold',color='k',backgroundcolor=titlecolor,fontsize=8)
    ax.axis('off')

plt.tight_layout()

plt.savefig('figure20_code_challenge_34.png')
plt.show()
files.download('figure20_code_challenge_34.png')
